### Event Loop

In [ ]:
import asyncio

loop = asyncio.get_running_loop()
print(f"running loop: {loop}")
print(f"is_running:   {loop.is_running()}")

### Coroutines & await

In [ ]:
import asyncio
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    time.sleep(brew)
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

In [ ]:
no_result = make_item()
no_result

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

In [ ]:
no_result = make_item()
no_result

**Sequential**

In [ ]:
start_time = time.monotonic()

drink = await make_item("coffee", 2.0)
food = await make_item("toast", 1.5)

print(f"sequential: {drink} + {food} — took {time.monotonic() - start_time:.2f}s")

**Concurrent with `gather()`**

In [ ]:
start_time = time.monotonic()

drink, food = await asyncio.gather(make_item("coffee", 2.0), make_item("toast", 1.5))

print(f"gather: {drink} + {food} — took {time.monotonic() - start_time:.2f}s")

**Five coffees at once**

In [ ]:
start_time = time.monotonic()

drinks = await asyncio.gather(*(make_item("coffee", 2.0) for _ in range(100000)))

print(f"five coffees: {len(drinks)} drinks — took {time.monotonic() - start_time:.2f}s")

---
- `await` pauses; sequential `await`s add up.
- `gather()` runs them at the same time — total ≈ slowest one.